# Conteo de registros H y M

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 1–11.

Versión para completar. Puedes trabajar en este cuaderno en lugar de `alumno.py`. Sustituye los `NotImplementedError` por tu código. Las celdas con tareas pendientes se detendrán hasta que las completes.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Conteo de registros por categoría H/M

El objetivo consiste en contar cuántos registros contienen la etiqueta H y cuántos la etiqueta M. La categoría se lee del último campo. El nombre no determina esa etiqueta.

Trabaja en `alumno.py`. Necesitas Python 3.10 o posterior y su biblioteca estándar. Los imports y los datos ya están preparados. Sustituye cada `NotImplementedError` por tu implementación. Ver ese error antes de completar una función es normal en esta plantilla.

## Datos y alcance

Hay tres ficheros con tres registros cada uno. Los tres registros de `1800.txt` reproducen el ejemplo visible de la diapositiva 3. Los otros seis registros son ficticios, creados para esta práctica. El resultado de los nueve registros corresponde a esta práctica, no a un conjunto completo mostrado en la presentación.

`FICHEROS` contiene los datos dentro del propio programa. Los archivos de `datos/` son su copia en UTF-8 con saltos de línea LF. No necesitas leerlos del disco para ejecutar el programa.

Cada fichero representa un split lógico didáctico. El recorrido se ejecutará de forma local y serial. H/M son las etiquetas de este conjunto didáctico. Contamos apariciones de registros, sin deduplicar personas entre ficheros.

## Trabajo paso a paso

1. Lee los nueve registros y calcula a mano el resultado esperado. Anota qué campo debe ser la clave intermedia para poder sumar por categoría.
2. Completa `map_registro`. Separa los campos y comprueba que hay un nombre no vacío, una sola coma y una categoría válida. Emite un par por registro. Pistas: `rpartition` y `strip`.
3. Completa `shuffle`. Recorre los pares emitidos y construye una lista de valores por clave. Conserva todos los valores repetidos. Pista: `defaultdict`.
4. Completa `reduce_categoria`. Escribe un bucle que acumule los valores de un grupo y devuelve el par de salida. Después contrasta tu acumulación con `sum`.
5. Completa `ejecutar_mapreduce`. Escribe los bucles que recorren los ficheros y sus líneas, recoge las emisiones, agrupa y reduce. Si un registro falla, informa del fichero y la línea. Pistas: `splitlines` y `enumerate`.
6. Completa `conteo_secuencial` con un recorrido directo que actualice los contadores sin construir emisiones ni grupos.
7. Completa `mostrar_demostracion`: muestra las nueve emisiones Map, los grupos y los dos resultados Reduce. Compara los métodos y comprueba que no se pierde ningún registro.

## Evidencias que debes obtener

- Nueve emisiones Map, una por registro.
- Dos grupos, uno por categoría, con todos sus valores.
- Un diccionario final con el conteo que hayas calculado para cada categoría.
- El mismo resultado mediante el conteo secuencial.
- Igualdad entre registros, emisiones Map y valores agrupados. Comprueba que `H + M = 9` usando los conteos que hayas obtenido.

Un registro vacío, uno sin coma, uno sin nombre o uno con categoría distinta de H/M debe provocar un error claro. No lo ignores silenciosamente. Los saltos LF que cierran los registros no forman un registro adicional.

## Relación con la presentación

Las diapositivas 7–9 usan el nombre como clave intermedia. La diapositiva 10 compara después la clave con H/M. Explica por qué esas dos decisiones no encajan. Para este objetivo la clave intermedia es la categoría, que es precisamente aquello por lo que se quiere contar.

## Qué representa la simulación

Las funciones muestran la transformación Map, la agrupación por clave y la reducción. El programa no ejecuta Hadoop ni reproduce HDFS, la transferencia por red, collect, spill, merge, planificación o tolerancia a fallos. No se deben extraer conclusiones de rendimiento distribuido a partir de esta ejecución local.


## Código 1 Presentación del bloque


In [6]:
"""Plantilla intermedia: conteo de registros por categoría H/M.

Python 3.10 o posterior. Solo biblioteca estándar.
Completa las funciones sustituyendo cada NotImplementedError.
Los tres primeros registros son los del ejemplo de la diapositiva 3.
Los otros seis son ficticios y se han añadido para esta práctica.
Cada fichero representa un split lógico didáctico. La ejecución es serial
y local. Esta plantilla no ejecuta Hadoop.
"""


'Plantilla intermedia: conteo de registros por categoría H/M.\n\nPython 3.10 o posterior. Solo biblioteca estándar.\nCompleta las funciones sustituyendo cada NotImplementedError.\nLos tres primeros registros son los del ejemplo de la diapositiva 3.\nLos otros seis son ficticios y se han añadido para esta práctica.\nCada fichero representa un split lógico didáctico. La ejecución es serial\ny local. Esta plantilla no ejecuta Hadoop.\n'

## Código 2 Imports y datos preparados


In [7]:
from collections import defaultdict


FICHEROS: dict[str, str] = {
    "1800.txt": (
        "Manuel García García, H\n"
        "Juan Belmonte Pérez, H\n"
        "María Selas Rodríguez, M\n"
    ),
    "1801.txt": (
        "Lucía Martín Pérez, M\n"
        "Diego Ramos Díaz, H\n"
        "Elena Suárez León, M\n"
    ),
    "1802.txt": (
        "Pablo Torres Gil, H\n"
        "Nora Medina Ruiz, M\n"
        "Iván Cabrera Soto, H\n"
    ),
}

CATEGORIAS = ("H", "M")


## Código 3 Map: parseo, validación y emisión


In [8]:
def map_registro(linea: str) -> tuple[str, int]:
    """Recibe una línea sin terminador y emite (categoría, 1).

    TODO: separa nombre y categoría. Rechaza nombres vacíos, registros
    sin una única coma, saltos de línea internos y categorías inválidas.
    Lee H/M del campo final. No deduzcas la categoría a partir del nombre.
    Pista: investiga rpartition y strip.
    """
    if "\n" in linea or "\r" in linea:
        raise ValueError(f"Salto de línea interno en el registro: {linea!r}")

    if linea.count(",") != 1:
        raise ValueError(f"El registro debe contener exactamente una coma: {linea!r}")

    nombre, sep, categoria = linea.rpartition(",")
    nombre = nombre.strip()
    categoria = categoria.strip()

    if not nombre:
        raise ValueError(f"El nombre no puede estar vacío: {linea!r}")
    if categoria not in CATEGORIAS:
        raise ValueError(f"Categoría inválida {categoria!r}. Permitidas: {CATEGORIAS}")

    return (categoria, 1)


## Código 4 Shuffle: agrupación sin perder repeticiones


In [9]:
def shuffle(pares: list[tuple[str, int]]) -> dict[str, list[int]]:
    """Recibe emisiones Map y devuelve una lista de valores por clave.

    TODO: escribe el recorrido de pares y la construcción de los grupos.
    Cada 1 debe conservarse, aunque otros valores sean iguales.
    Pista: defaultdict puede facilitar la creación de listas vacías.
    """
    grupos: dict[str, list[int]] = defaultdict(list)
    for clave, valor in pares:
        grupos[clave].append(valor)
    return dict(grupos)


## Código 5 Reduce: cálculo por categoría


In [10]:
def reduce_categoria(clave: str, valores: list[int]) -> tuple[str, int]:
    """Devuelve (categoría, total) para un único grupo.

    TODO: escribe la acumulación de los valores y devuelve el par de salida.
    Pista: resuélvelo primero con un bucle. Después puedes compararlo con sum.
    """
    total = 0
    for v in valores:
        total += v

    assert total == sum(valores), "La acumulación no coincide con sum()"
    return (clave, total)


## Código 6 Coordinación de las fases


In [11]:
def ejecutar_mapreduce(ficheros: dict[str, str]) -> dict[str, int]:
    """Devuelve el conteo de las categorías presentes en los registros.

    TODO: recorre los ficheros y sus líneas, llama a Map y recoge sus pares.
    Después agrupa y aplica Reduce a cada clave. Construye el resultado.
    Los errores deben indicar fichero y número de línea, sin ignorarlos.
    Pista: splitlines, enumerate y los contratos de las funciones anteriores.
    """
    emisiones: list[tuple[str, int]] = []

    # Fase Map: recorrer ficheros y líneas
    for nombre_fichero, contenido in ficheros.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            try:
                par = map_registro(linea)
                emisiones.append(par)
            except Exception as e:
                raise ValueError(
                    f"Error en fichero {nombre_fichero!r}, línea {num_linea}: {e}"
                ) from e

    # Fase Shuffle: agrupar por clave
    grupos = shuffle(emisiones)

    # Fase Reduce: reducir cada grupo a su total
    resultado: dict[str, int] = {}
    for clave, valores in grupos.items():
        cat, total = reduce_categoria(clave, valores)
        resultado[cat] = total

    return resultado


## Código 7 Contraste secuencial


In [12]:
def conteo_secuencial(ficheros: dict[str, str]) -> dict[str, int]:
    """Cuenta directamente los registros H/M sin crear listas de emisiones.

    TODO: recorre y valida los datos. Actualiza un contador por categoría.
    Este recorrido permitirá contrastar tu resultado MapReduce.
    """
    contadores: dict[str, int] = {cat: 0 for cat in CATEGORIAS}

    for nombre_fichero, contenido in ficheros.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            if "\n" in linea or "\r" in linea:
                raise ValueError(
                    f"Error en {nombre_fichero!r}:{num_linea}: salto de línea interno"
                )
            if linea.count(",") != 1:
                raise ValueError(
                    f"Error en {nombre_fichero!r}:{num_linea}: el registro debe tener una coma"
                )

            nombre, sep, categoria = linea.rpartition(",")
            nombre = nombre.strip()
            categoria = categoria.strip()

            if not nombre:
                raise ValueError(
                    f"Error en {nombre_fichero!r}:{num_linea}: el nombre no puede estar vacío"
                )
            if categoria not in CATEGORIAS:
                raise ValueError(
                    f"Error en {nombre_fichero!r}:{num_linea}: categoría inválida {categoria!r}"
                )

            contadores[categoria] += 1

    return contadores


## Código 8 Mostrar las fases y comprobar el resultado


In [ ]:
def mostrar_demostracion() -> None:
    """Construye una traza de los datos preparados.

    TODO: muestra una emisión por registro, los grupos y cada salida Reduce.
    Ejecuta ambos métodos y comprueba su igualdad. Comprueba también que el
    número de registros, emisiones y valores agrupados coincide con H + M.
    Calcula los conteos de H y M. Su suma debe ser igual a los 9 registros.
    """
    print("=" * 60)
    print("FASE 1: MAP - Emisiones por registro (9 emisiones)")
    print("=" * 60)
    emisiones: list[tuple[str, int]] = []
    total_registros = 0
    for nombre_fichero, contenido in FICHEROS.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            total_registros += 1
            par = map_registro(linea)
            emisiones.append(par)
            print(f"[{nombre_fichero}:{num_linea}] {linea!r:28} -> {par}")

    print("\n" + "=" * 60)
    print("FASE 2: SHUFFLE - Agrupación por clave intermedia")
    print("=" * 60)
    grupos = shuffle(emisiones)
    for clave in CATEGORIAS:
        valores = grupos.get(clave, [])
        print(f"Grupo {clave!r}: {valores}")

    print("\n" + "=" * 60)
    print("FASE 3: REDUCE - Reducción y suma por categoría")
    print("=" * 60)
    resultado_mapreduce: dict[str, int] = {}
    for clave in CATEGORIAS:
        valores = grupos.get(clave, [])
        cat, total = reduce_categoria(clave, valores)
        resultado_mapreduce[cat] = total
        print(f"Reduce {cat!r} -> ({cat!r}, {total})")

    print(f"\nDiccionario final MapReduce: {resultado_mapreduce}")

    print("\n" + "=" * 60)
    print("FASE 4: CONTRASTE CON CONTEO SECUENCIAL")
    print("=" * 60)
    resultado_secuencial = conteo_secuencial(FICHEROS)
    print(f"Diccionario final Secuencial: {resultado_secuencial}")

    print("\n" + "=" * 60)
    print("EVIDENCIAS Y COMPROBACIONES")
    print("=" * 60)
    total_emisiones = len(emisiones)
    total_valores_agrupados = sum(len(v) for v in grupos.values())
    conteo_h = resultado_mapreduce.get("H", 0)
    conteo_m = resultado_mapreduce.get("M", 0)
    suma_hm = conteo_h + conteo_m

    print(f"1. Total registros leídos:       {total_registros}")
    print(f"2. Total emisiones Map:          {total_emisiones}")
    print(f"3. Total valores agrupados:      {total_valores_agrupados}")
    print(f"4. Conteo H ({conteo_h}) + Conteo M ({conteo_m}): {suma_hm}")

    assert total_registros == 9, f"Se esperaban 9 registros, hay {total_registros}"
    assert total_emisiones == 9, f"Se esperaban 9 emisiones Map, hay {total_emisiones}"
    assert total_valores_agrupados == 9, f"Se esperaban 9 valores agrupados, hay {total_valores_agrupados}"
    assert suma_hm == 9, f"H + M debe ser 9, pero da {suma_hm}"
    assert resultado_mapreduce == resultado_secuencial, (
        f"MapReduce {resultado_mapreduce} y Secuencial {resultado_secuencial} deben coincidir"
    )

    print("\n[OK] Todo se ha realizado con exito")


## Código 9 Ejecutar tu demostración


In [14]:
if __name__ == "__main__":
    mostrar_demostracion()


FASE 1: MAP - Emisiones por registro (9 emisiones)
[1800.txt:1] 'Manuel García García, H'    -> ('H', 1)
[1800.txt:2] 'Juan Belmonte Pérez, H'     -> ('H', 1)
[1800.txt:3] 'María Selas Rodríguez, M'   -> ('M', 1)
[1801.txt:1] 'Lucía Martín Pérez, M'      -> ('M', 1)
[1801.txt:2] 'Diego Ramos Díaz, H'        -> ('H', 1)
[1801.txt:3] 'Elena Suárez León, M'       -> ('M', 1)
[1802.txt:1] 'Pablo Torres Gil, H'        -> ('H', 1)
[1802.txt:2] 'Nora Medina Ruiz, M'        -> ('M', 1)
[1802.txt:3] 'Iván Cabrera Soto, H'       -> ('H', 1)

FASE 2: SHUFFLE - Agrupación por clave intermedia
Grupo 'H': [1, 1, 1, 1, 1]
Grupo 'M': [1, 1, 1, 1]

FASE 3: REDUCE - Reducción y suma por categoría
Reduce 'H' -> ('H', 5)
Reduce 'M' -> ('M', 4)

Diccionario final MapReduce: {'H': 5, 'M': 4}

FASE 4: CONTRASTE CON CONTEO SECUENCIAL
Diccionario final Secuencial: {'H': 5, 'M': 4}

EVIDENCIAS Y COMPROBACIONES
1. Total registros leídos:       9
2. Total emisiones Map:          9
3. Total valores agrupados:     

## Explicación de tus resultados

Anota aquí tus conclusiones y las respuestas que pide el guion.
